In [20]:
#Импорт библиотек
import pandas as pd
import numpy as np
import re
from google.colab import files

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)

#Загрузка файлов
uploaded = files.upload()

prol = pd.read_csv('prolongations.csv')
fin = pd.read_csv('financial_data.csv')

print('prolongations:', prol.shape)
print('financial_data:', fin.shape)

display(prol.head())
display(fin.head())

Saving prolongations.csv to prolongations (1).csv
Saving financial_data.csv to financial_data (1).csv
prolongations: (477, 3)
financial_data: (451, 19)


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [23]:
#Справочники месяцев
month_map = {
    'январь': 1,
    'февраль': 2,
    'март': 3,
    'апрель': 4,
    'май': 5,
    'июнь': 6,
    'июль': 7,
    'август': 8,
    'сентябрь': 9,
    'октябрь': 10,
    'ноябрь': 11,
    'декабрь': 12
}

month_map_title = {
    'Январь': 1,
    'Февраль': 2,
    'Март': 3,
    'Апрель': 4,
    'Май': 5,
    'Июнь': 6,
    'Июль': 7,
    'Август': 8,
    'Сентябрь': 9,
    'Октябрь': 10,
    'Ноябрь': 11,
    'Декабрь': 12
}

def parse_month_text(text):
    """
    Преобразует строку вида 'ноябрь 2022' или 'Ноябрь 2022'
    в Timestamp('2022-11-01')
    """
    if pd.isna(text):
        return pd.NaT

    text = str(text).strip()
    parts = text.split()

    if len(parts) != 2:
        return pd.NaT

    month_name, year = parts

    if month_name in month_map:
        month_num = month_map[month_name]
    elif month_name in month_map_title:
        month_num = month_map_title[month_name]
    else:
        return pd.NaT

    return pd.Timestamp(year=int(year), month=month_num, day=1)

In [24]:
#Базовая подготовка таблиц
prol = prol.copy()
fin = fin.copy()

prol.columns = [c.strip() for c in prol.columns]
fin.columns = [c.strip() for c in fin.columns]

prol['end_month_dt'] = prol['month'].apply(parse_month_text)

print('Колонки prolongations:')
print(prol.columns.tolist())
print()

print('Колонки financial_data:')
print(fin.columns.tolist())

Колонки prolongations:
['id', 'month', 'AM', 'end_month_dt']

Колонки financial_data:
['id', 'Причина дубля', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024', 'Account']


In [25]:
#Определяем месячные колонки
def is_month_col(col):
    return pd.notna(parse_month_text(col))

month_cols = [col for col in fin.columns if is_month_col(col)]

month_df = pd.DataFrame({
    'month_col': month_cols,
    'month_dt': [parse_month_text(col) for col in month_cols]
}).sort_values('month_dt').reset_index(drop=True)

month_cols = month_df['month_col'].tolist()
month_dt_to_col = dict(zip(month_df['month_dt'], month_df['month_col']))

print('Месячные колонки:')
print(month_cols)

Месячные колонки:
['Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024']


In [27]:
#Функции для обработки значений
def normalize_cell(x):
    """
    Унифицируем значения в ячейках:
    - числа оставляем строкой для дальнейшего парсинга
    - текстовые спецзначения приводим к нижнему регистру
    - пустые значения -> np.nan
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == '':
        return np.nan

    x_low = x.lower()
    if x_low in ['в ноль', 'стоп', 'end']:
        return x_low

    return x

def parse_amount(x):
    """
    Парсим денежное значение из строки формата:
    '36 220,00', '0,00', '329 460,000' и т.д.
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x in ['в ноль', 'стоп', 'end', '']:
        return np.nan

    x = x.replace('\xa0', '')
    x = x.replace(' ', '')
    x = x.replace(',', '.')

    try:
        return float(x)
    except:
        return np.nan

for col in month_cols:
    fin[col] = fin[col].apply(normalize_cell)

In [28]:
#Сводим financial_data до уровня id
# В financial_data один id может встречаться несколько раз:
# первая/вторая часть оплаты, доп работы, изменение ЮЛ и т.д.

special_priority = {
    'stop': 3,
    'end': 3,
    'в ноль': 2
}

def aggregate_month_values(series):
    """
    Правила агрегации месячных значений по одному id:
    1. Если есть хотя бы одно 'stop' или 'end' -> возвращаем 'stop' / 'end'
       (для наших расчётов это сигнал на исключение).
    2. Иначе суммируем все числовые значения.
    3. Если чисел нет, но есть 'в ноль' -> возвращаем 'в ноль'
    4. Иначе NaN
    """
    values = [v for v in series if pd.notna(v)]
    if len(values) == 0:
        return np.nan

    values_low = [str(v).strip().lower() for v in values]

    if 'stop' in values_low:
        return 'stop'
    if 'end' in values_low:
        return 'end'

    nums = [parse_amount(v) for v in values]
    nums = [v for v in nums if pd.notna(v)]

    if len(nums) > 0:
        return sum(nums)

    if 'в ноль' in values_low:
        return 'в ноль'

    return np.nan

agg_dict = {col: aggregate_month_values for col in month_cols}
agg_dict['Account'] = 'first'

fin_agg = fin.groupby('id', as_index=False).agg(agg_dict)

print('financial_data после агрегации:', fin_agg.shape)
display(fin_agg.head())

financial_data после агрегации: (314, 18)


,id,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,15,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Иванова Мария Сергеевна
1,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Иванова Мария Сергеевна
2,31,55100.0,55100.0,NaN,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,44775.0,46200.0,Васильев Артем Александрович
3,39,137700.0,137700.0,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,149206.5,NaN,NaN,Попова Екатерина Николаевна
4,42,36220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [29]:
#Объединяем таблицы
# Менеджер из prolongations.csv является первичным по ТЗ

df = prol.merge(fin_agg, on='id', how='left', suffixes=('', '_fin'))

# Единое имя менеджера
df['manager'] = df['AM']

print('Объединённая таблица:', df.shape)
display(df[['id', 'month', 'end_month_dt', 'AM', 'Account', 'manager']].head())

Объединённая таблица: (477, 22)


,id,month,end_month_dt,AM,Account,manager
0,42,ноябрь 2022,2022-11-01,Васильев Артем Александрович,Васильев Артем Александрович,Васильев Артем Александрович
1,453,ноябрь 2022,2022-11-01,Васильев Артем Александрович,Васильев Артем Александрович,Васильев Артем Александрович
2,548,ноябрь 2022,2022-11-01,Михайлов Андрей Сергеевич,Михайлов Андрей Сергеевич,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,2022-11-01,Соколова Анастасия Викторовна,Иванова Мария Сергеевна,Соколова Анастасия Викторовна
4,429,ноябрь 2022,2022-11-01,Соколова Анастасия Викторовна,Соколова Анастасия Викторовна,Соколова Анастасия Викторовна


In [31]:
#Функции бизнес-логики
def get_month_col(month_dt):
    return month_dt_to_col.get(pd.Timestamp(month_dt))

def has_stop_or_end_before_or_at(row, end_month_dt):
    """
    Если у проекта есть stop/end в последний месяц реализации или раньше,
    исключаем проект из пролонгаций.
    """
    relevant_months = month_df[month_df['month_dt'] <= end_month_dt]['month_col'].tolist()

    for col in relevant_months:
        val = row.get(col, np.nan)
        if pd.isna(val):
            continue
        if str(val).strip().lower() in ['stop', 'end']:
            return True
    return False

def get_value_in_month(row, month_dt):
    """
    Возвращает сырое значение по месяцу
    """
    col = get_month_col(month_dt)
    if col is None:
        return np.nan
    return row.get(col, np.nan)

def get_numeric_in_month(row, month_dt):
    """
    Возвращает число, если в месяце есть число.
    Иначе np.nan
    """
    return parse_amount(get_value_in_month(row, month_dt))

def get_last_real_shipment(row, end_month_dt):
    """
    База для знаменателя:
    - если в последний месяц есть число -> берём его
    - если 'в ноль' -> берём предыдущий месяц с числом
    - если stop/end -> проект должен быть исключён ранее
    """
    val = get_value_in_month(row, end_month_dt)

    if pd.isna(val):
        return np.nan

    val_low = str(val).strip().lower()
    val_num = parse_amount(val)

    if pd.notna(val_num):
        return val_num

    if val_low == 'в ноль':
        prev_months = month_df[month_df['month_dt'] < end_month_dt].sort_values('month_dt', ascending=False)
        for _, r in prev_months.iterrows():
            prev_val = parse_amount(row.get(r['month_col'], np.nan))
            if pd.notna(prev_val):
                return prev_val

    return np.nan

def has_positive_shipment(row, month_dt):
    """
    Считаем признаком пролонгации наличие положительной суммы отгрузки
    """
    val = get_numeric_in_month(row, month_dt)
    return pd.notna(val) and val > 0

In [32]:
#Подготовка расчётной базы
calc = df.copy()
#Валидный месяц завершения
calc = calc[calc['end_month_dt'].notna()].copy()
#Исключаем stop/end
calc['exclude_stop_end'] = calc.apply(
    lambda row: has_stop_or_end_before_or_at(row, row['end_month_dt']),
    axis=1
)

calc = calc[calc['exclude_stop_end'] == False].copy()
#База = отгрузка в последний месяц реализации
calc['base_last_shipment'] = calc.apply(
    lambda row: get_last_real_shipment(row, row['end_month_dt']),
    axis=1
)

calc = calc[calc['base_last_shipment'].notna()].copy()

print('Расчётная база:', calc.shape)
display(calc[['id', 'manager', 'month', 'end_month_dt', 'base_last_shipment']].head(20))

Расчётная база: (404, 24)


,id,manager,month,end_month_dt,base_last_shipment
0,42,Васильев Артем Александрович,ноябрь 2022,2022-11-01,36220.0
2,548,Михайлов Андрей Сергеевич,ноябрь 2022,2022-11-01,674000.0
3,87,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,70050.0
4,429,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,30280.0
5,600,Васильев Артем Александрович,ноябрь 2022,2022-11-01,281417.0
8,665,Васильев Артем Александрович,ноябрь 2022,2022-11-01,10000.0
9,586,Иванова Мария Сергеевна,ноябрь 2022,2022-11-01,52240.0
10,637,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,38045.0
12,578,Попова Екатерина Николаевна,ноябрь 2022,2022-11-01,82800.0
14,671,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,28000.0


In [33]:
#Пролонгация в 1-й и 2-й месяц
calc['report_month_m1'] = calc['end_month_dt'] + pd.DateOffset(months=1)
calc['report_month_m2'] = calc['end_month_dt'] + pd.DateOffset(months=2)

calc['shipment_m1'] = calc.apply(lambda row: get_numeric_in_month(row, row['report_month_m1']), axis=1)
calc['shipment_m2'] = calc.apply(lambda row: get_numeric_in_month(row, row['report_month_m2']), axis=1)

calc['prolonged_m1'] = calc['shipment_m1'].fillna(0) > 0
calc['prolonged_m2'] = (calc['shipment_m1'].fillna(0) <= 0) & (calc['shipment_m2'].fillna(0) > 0)

display(calc[['id', 'manager', 'end_month_dt', 'base_last_shipment', 'shipment_m1', 'shipment_m2', 'prolonged_m1', 'prolonged_m2']].head(20))

,id,manager,end_month_dt,base_last_shipment,shipment_m1,shipment_m2,prolonged_m1,prolonged_m2
0,42,Васильев Артем Александрович,2022-11-01,36220.0,NaN,NaN,False,False
2,548,Михайлов Андрей Сергеевич,2022-11-01,674000.0,674000.00,674000.00,True,False
3,87,Соколова Анастасия Викторовна,2022-11-01,70050.0,NaN,73380.00,False,True
4,429,Соколова Анастасия Викторовна,2022-11-01,30280.0,35580.00,35830.00,True,False
5,600,Васильев Артем Александрович,2022-11-01,281417.0,175100.00,267220.00,True,False
8,665,Васильев Артем Александрович,2022-11-01,10000.0,NaN,NaN,False,False
9,586,Иванова Мария Сергеевна,2022-11-01,52240.0,NaN,NaN,False,False
10,637,Соколова Анастасия Викторовна,2022-11-01,38045.0,NaN,NaN,False,False
12,578,Попова Екатерина Николаевна,2022-11-01,82800.0,NaN,NaN,False,False
14,671,Соколова Анастасия Викторовна,2022-11-01,28000.0,105805.00,108230.00,True,False


In [34]:
#Коэффициент первого месяца
m1 = calc[calc['report_month_m1'].dt.year == 2023].copy()

m1_manager = (
    m1.groupby(['manager', 'report_month_m1'], as_index=False)
      .agg(
          base_sum_m1=('base_last_shipment', 'sum'),
          prolong_sum_m1=('shipment_m1', lambda s: s.fillna(0).sum()),
          projects_cnt_m1=('id', 'count'),
          prolonged_cnt_m1=('prolonged_m1', 'sum')
      )
)

m1_manager['coef_m1'] = m1_manager['prolong_sum_m1'] / m1_manager['base_sum_m1']

display(m1_manager.head(20))

,manager,report_month_m1,base_sum_m1,prolong_sum_m1,projects_cnt_m1,prolonged_cnt_m1,coef_m1
0,Васильев Артем Александрович,2023-01-01,1693097.68,941761.60,12,6,0.556236
1,Васильев Артем Александрович,2023-02-01,827225.00,874015.00,4,2,1.056563
2,Васильев Артем Александрович,2023-03-01,991467.42,621897.55,9,5,0.627250
3,Васильев Артем Александрович,2023-04-01,1601033.35,562360.00,13,5,0.351248
4,Васильев Артем Александрович,2023-05-01,2277723.50,1010023.00,10,6,0.443435
5,Васильев Артем Александрович,2023-06-01,390286.00,151933.28,4,2,0.389287
6,Васильев Артем Александрович,2023-07-01,891755.60,521691.00,9,5,0.585016
7,Васильев Артем Александрович,2023-08-01,649000.00,309900.00,5,3,0.477504
8,Васильев Артем Александрович,2023-09-01,1361540.74,234774.00,8,3,0.172433
9,Васильев Артем Александрович,2023-10-01,401370.00,355915.00,6,1,0.886750


In [35]:
#Коэффициент второго месяца
m2 = calc[calc['report_month_m2'].dt.year == 2023].copy()
#Во 2-й месяц входят только проекты, которые НЕ пролонгировались в 1-й
m2 = m2[m2['prolonged_m1'] == False].copy()

m2_manager = (
    m2.groupby(['manager', 'report_month_m2'], as_index=False)
      .agg(
          base_sum_m2=('base_last_shipment', 'sum'),
          prolong_sum_m2=('shipment_m2', lambda s: s.fillna(0).sum()),
          projects_cnt_m2=('id', 'count'),
          prolonged_cnt_m2=('prolonged_m2', 'sum')
      )
)

m2_manager['coef_m2'] = m2_manager['prolong_sum_m2'] / m2_manager['base_sum_m2']

display(m2_manager.head(20))

,manager,report_month_m2,base_sum_m2,prolong_sum_m2,projects_cnt_m2,prolonged_cnt_m2,coef_m2
0,Васильев Артем Александрович,2023-01-01,260862.00,0.00,4,0,0.000000
1,Васильев Артем Александрович,2023-02-01,891335.00,166275.00,6,2,0.186546
2,Васильев Артем Александрович,2023-03-01,95750.00,67774.08,2,1,0.707823
3,Васильев Артем Александрович,2023-04-01,364780.00,40200.00,4,1,0.110203
4,Васильев Артем Александрович,2023-05-01,1004848.80,0.00,8,0,0.000000
5,Васильев Артем Александрович,2023-06-01,678208.50,38632.50,4,1,0.056963
6,Васильев Артем Александрович,2023-07-01,230686.00,0.00,2,0,0.000000
7,Васильев Артем Александрович,2023-08-01,492017.60,0.00,4,0,0.000000
8,Васильев Артем Александрович,2023-09-01,274500.00,0.00,2,0,0.000000
9,Васильев Артем Александрович,2023-10-01,1135125.00,82020.00,5,1,0.072256


In [36]:
#Единый отчёт по менеджерам за месяцы 2023
monthly_manager = pd.merge(
    m1_manager,
    m2_manager,
    left_on=['manager', 'report_month_m1'],
    right_on=['manager', 'report_month_m2'],
    how='outer'
)

monthly_manager['report_month'] = monthly_manager['report_month_m1'].combine_first(monthly_manager['report_month_m2'])

monthly_manager.drop(columns=['report_month_m1', 'report_month_m2'], inplace=True)

fill_zero_cols = [
    'base_sum_m1', 'prolong_sum_m1', 'projects_cnt_m1', 'prolonged_cnt_m1', 'coef_m1',
    'base_sum_m2', 'prolong_sum_m2', 'projects_cnt_m2', 'prolonged_cnt_m2', 'coef_m2'
]

for col in fill_zero_cols:
    if col in monthly_manager.columns:
        monthly_manager[col] = monthly_manager[col].fillna(0)

monthly_manager = monthly_manager.sort_values(['manager', 'report_month']).reset_index(drop=True)

monthly_manager['report_month_str'] = monthly_manager['report_month'].dt.strftime('%Y-%m')

display(monthly_manager.head(30))

,manager,base_sum_m1,prolong_sum_m1,projects_cnt_m1,prolonged_cnt_m1,coef_m1,base_sum_m2,prolong_sum_m2,projects_cnt_m2,prolonged_cnt_m2,coef_m2,report_month,report_month_str
0,Васильев Артем Александрович,1693097.68,941761.60,12.0,6.0,0.556236,260862.00,0.00,4.0,0.0,0.000000,2023-01-01,2023-01
1,Васильев Артем Александрович,827225.00,874015.00,4.0,2.0,1.056563,891335.00,166275.00,6.0,2.0,0.186546,2023-02-01,2023-02
2,Васильев Артем Александрович,991467.42,621897.55,9.0,5.0,0.627250,95750.00,67774.08,2.0,1.0,0.707823,2023-03-01,2023-03
3,Васильев Артем Александрович,1601033.35,562360.00,13.0,5.0,0.351248,364780.00,40200.00,4.0,1.0,0.110203,2023-04-01,2023-04
4,Васильев Артем Александрович,2277723.50,1010023.00,10.0,6.0,0.443435,1004848.80,0.00,8.0,0.0,0.000000,2023-05-01,2023-05
5,Васильев Артем Александрович,390286.00,151933.28,4.0,2.0,0.389287,678208.50,38632.50,4.0,1.0,0.056963,2023-06-01,2023-06
6,Васильев Артем Александрович,891755.60,521691.00,9.0,5.0,0.585016,230686.00,0.00,2.0,0.0,0.000000,2023-07-01,2023-07
7,Васильев Артем Александрович,649000.00,309900.00,5.0,3.0,0.477504,492017.60,0.00,4.0,0.0,0.000000,2023-08-01,2023-08
8,Васильев Артем Александрович,1361540.74,234774.00,8.0,3.0,0.172433,274500.00,0.00,2.0,0.0,0.000000,2023-09-01,2023-09
9,Васильев Артем Александрович,401370.00,355915.00,6.0,1.0,0.886750,1135125.00,82020.00,5.0,1.0,0.072256,2023-10-01,2023-10


In [37]:
#По отделу за каждый месяц
monthly_department = (
    monthly_manager.groupby('report_month', as_index=False)
    .agg(
        base_sum_m1=('base_sum_m1', 'sum'),
        prolong_sum_m1=('prolong_sum_m1', 'sum'),
        projects_cnt_m1=('projects_cnt_m1', 'sum'),
        prolonged_cnt_m1=('prolonged_cnt_m1', 'sum'),
        base_sum_m2=('base_sum_m2', 'sum'),
        prolong_sum_m2=('prolong_sum_m2', 'sum'),
        projects_cnt_m2=('projects_cnt_m2', 'sum'),
        prolonged_cnt_m2=('prolonged_cnt_m2', 'sum')
    )
)

monthly_department['coef_m1'] = monthly_department['prolong_sum_m1'] / monthly_department['base_sum_m1']
monthly_department['coef_m2'] = monthly_department['prolong_sum_m2'] / monthly_department['base_sum_m2']
monthly_department['report_month_str'] = monthly_department['report_month'].dt.strftime('%Y-%m')

display(monthly_department)

,report_month,base_sum_m1,prolong_sum_m1,projects_cnt_m1,prolonged_cnt_m1,base_sum_m2,prolong_sum_m2,projects_cnt_m2,prolonged_cnt_m2,coef_m1,coef_m2,report_month_str
0,2023-01-01,5986766.86,2677021.61,50.0,23.0,545022.00,73380.00,9.0,1.0,0.447156,0.134637,2023-01
1,2023-02-01,3400075.50,1880945.00,17.0,6.0,2890042.76,262620.00,27.0,4.0,0.553207,0.090871,2023-02
2,2023-03-01,1899332.77,1271739.40,21.0,13.0,1540840.50,122999.08,11.0,2.0,0.669572,0.079826,2023-03
3,2023-04-01,4157742.20,1501923.35,33.0,16.0,627735.00,57940.00,8.0,2.0,0.361235,0.092300,2023-04
4,2023-05-01,3647436.50,2072660.75,23.0,17.0,2576703.80,0.00,17.0,0.0,0.568251,0.000000,2023-05
5,2023-06-01,1274625.53,281673.28,19.0,3.0,1015068.50,38632.50,6.0,1.0,0.220985,0.038059,2023-06
6,2023-07-01,2650982.85,1390411.00,28.0,14.0,1020325.53,69385.00,16.0,1.0,0.524489,0.068003,2023-07
7,2023-08-01,1936915.83,943678.88,21.0,12.0,1477499.85,52315.00,14.0,1.0,0.487207,0.035408,2023-08
8,2023-09-01,3306198.26,1086504.42,25.0,10.0,755739.63,0.00,9.0,0.0,0.328627,0.000000,2023-09
9,2023-10-01,3820835.81,3162695.19,38.0,21.0,2247364.88,245379.00,15.0,2.0,0.827750,0.109185,2023-10


In [38]:
#По менеджерам за весь 2023 год
yearly_manager = (
    monthly_manager.groupby('manager', as_index=False)
    .agg(
        year_base_sum_m1=('base_sum_m1', 'sum'),
        year_prolong_sum_m1=('prolong_sum_m1', 'sum'),
        year_projects_cnt_m1=('projects_cnt_m1', 'sum'),
        year_prolonged_cnt_m1=('prolonged_cnt_m1', 'sum'),
        year_base_sum_m2=('base_sum_m2', 'sum'),
        year_prolong_sum_m2=('prolong_sum_m2', 'sum'),
        year_projects_cnt_m2=('projects_cnt_m2', 'sum'),
        year_prolonged_cnt_m2=('prolonged_cnt_m2', 'sum')
    )
)

yearly_manager['year_coef_m1'] = yearly_manager['year_prolong_sum_m1'] / yearly_manager['year_base_sum_m1']
yearly_manager['year_coef_m2'] = yearly_manager['year_prolong_sum_m2'] / yearly_manager['year_base_sum_m2']

yearly_manager = yearly_manager.sort_values('year_coef_m1', ascending=False).reset_index(drop=True)

display(yearly_manager)

,manager,year_base_sum_m1,year_prolong_sum_m1,year_projects_cnt_m1,year_prolonged_cnt_m1,year_base_sum_m2,year_prolong_sum_m2,year_projects_cnt_m2,year_prolonged_cnt_m2,year_coef_m1,year_coef_m2
0,Петрова Анна Дмитриевна,98492.00,109442.52,1.0,1.0,0.00,0.00,0.0,0.0,1.111182,NaN
1,Смирнова Ольга Владимировна,2951204.59,2073563.50,40.0,21.0,803500.80,203825.00,14.0,4.0,0.702616,0.253671
2,Михайлов Андрей Сергеевич,3361833.59,2235422.29,21.0,14.0,1056004.68,0.00,7.0,0.0,0.664941,0.000000
3,Соколова Анастасия Викторовна,6862934.34,3974267.97,64.0,34.0,2876881.74,169725.00,31.0,3.0,0.579092,0.058996
4,Васильев Артем Александрович,11832147.79,5998164.43,88.0,42.0,5818845.40,482441.58,48.0,7.0,0.506938,0.082910
5,Кузнецов Михаил Иванович,982724.44,470182.98,11.0,7.0,167675.00,0.00,2.0,0.0,0.478448,0.000000
6,Попова Екатерина Николаевна,5265853.08,1885685.80,54.0,25.0,3335920.26,202499.00,29.0,3.0,0.358097,0.060703
7,Иванова Мария Сергеевна,4727657.16,1660095.55,40.0,15.0,2503864.75,0.00,26.0,0.0,0.351146,0.000000


In [39]:
#По отделу за 2023 год
yearly_department = pd.DataFrame({
    'year_base_sum_m1': [monthly_manager['base_sum_m1'].sum()],
    'year_prolong_sum_m1': [monthly_manager['prolong_sum_m1'].sum()],
    'year_projects_cnt_m1': [monthly_manager['projects_cnt_m1'].sum()],
    'year_prolonged_cnt_m1': [monthly_manager['prolonged_cnt_m1'].sum()],
    'year_base_sum_m2': [monthly_manager['base_sum_m2'].sum()],
    'year_prolong_sum_m2': [monthly_manager['prolong_sum_m2'].sum()],
    'year_projects_cnt_m2': [monthly_manager['projects_cnt_m2'].sum()],
    'year_prolonged_cnt_m2': [monthly_manager['prolonged_cnt_m2'].sum()]
})

yearly_department['year_coef_m1'] = yearly_department['year_prolong_sum_m1'] / yearly_department['year_base_sum_m1']
yearly_department['year_coef_m2'] = yearly_department['year_prolong_sum_m2'] / yearly_department['year_base_sum_m2']

display(yearly_department)

,year_base_sum_m1,year_prolong_sum_m1,year_projects_cnt_m1,year_prolonged_cnt_m1,year_base_sum_m2,year_prolong_sum_m2,year_projects_cnt_m2,year_prolonged_cnt_m2,year_coef_m1,year_coef_m2
0,36082846.99,18406825.04,319.0,159.0,16562692.63,1058490.58,157.0,17.0,0.510127,0.063908


In [40]:
#Выгрузка в Excel
monthly_manager_export = monthly_manager.copy()
monthly_department_export = monthly_department.copy()

monthly_manager_export.drop(columns=['report_month'], inplace=True)
monthly_department_export.drop(columns=['report_month'], inplace=True)

with pd.ExcelWriter('prolongation_report.xlsx', engine='openpyxl') as writer:
    monthly_manager_export.to_excel(writer, sheet_name='manager_monthly', index=False)
    monthly_department_export.to_excel(writer, sheet_name='department_monthly', index=False)
    yearly_manager.to_excel(writer, sheet_name='manager_yearly', index=False)
    yearly_department.to_excel(writer, sheet_name='department_yearly', index=False)

print('Файл prolongation_report.xlsx успешно сохранён')

Файл prolongation_report.xlsx успешно сохранён


In [19]:
from google.colab import files
files.download('prolongation_report.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>